# Concrete Crack Detection Using CNNs

In this notebook, I'm building CNN models to classify concrete surface images as either having cracks (Positive) or no cracks (Negative). I'll compare a custom CNN with transfer learning models (MobileNetV2 and ResNet50).

**Dataset:** [Surface Crack Detection](https://www.kaggle.com/datasets/arunrk7/surface-crack-detection)

**Task:** Binary classification — Crack vs No Crack

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import cv2
import warnings
import time
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2, ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.model_selection import train_test_split

import random

# setting seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

# white background for all plots
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


In [ ]:
def save_fig(name, dpi=150):
    """Save figure as both PNG (quick preview) and PDF (for Overleaf — no pixelation when zoomed).
    Professor requirement: use PDF figures in the report, never PNG → PDF conversions.
    """
    plt.savefig(f'{name}.png', dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.savefig(f'{name}.pdf', bbox_inches='tight', facecolor='white')
    print(f"Saved: {name}.png  |  {name}.pdf")


## 1. Loading and Exploring the Dataset

The dataset has two folders: Positive (crack) and Negative (no crack). Let me first look at what we're working with.

In [ ]:
# paths — update if running locally
data_dir = '/kaggle/input/datasets/arunrk7/surface-crack-detection'

positive_dir = os.path.join(data_dir, 'Positive')
negative_dir = os.path.join(data_dir, 'Negative')

# count images
pos_count = len(os.listdir(positive_dir))
neg_count = len(os.listdir(negative_dir))

print(f"Positive (Crack): {pos_count} images")
print(f"Negative (No Crack): {neg_count} images")
print(f"Total: {pos_count + neg_count} images")
print(f"\nClass balance ratio: {pos_count/neg_count:.2f}:1")
print("Dataset is perfectly balanced!")

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

def get_file_paths_and_labels(folder, label):
    paths = []
    labels = []
    for filename in sorted(os.listdir(folder)):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            paths.append(os.path.join(folder, filename))
            labels.append(label)
    return paths, labels

print("Collecting file paths (no images loaded into RAM)...")

pos_paths, pos_labels = get_file_paths_and_labels(positive_dir, 1)
neg_paths, neg_labels = get_file_paths_and_labels(negative_dir, 0)

all_paths = pos_paths + neg_paths
all_labels = pos_labels + neg_labels

print(f"Total image paths collected: {len(all_paths)}")
print(f"Crack (Positive):   {sum(all_labels)}")
print(f"No Crack (Negative): {len(all_labels) - sum(all_labels)}")
print(f"\nRAM used: ~{len(all_paths) * 80 / 1e6:.1f} MB (just file path strings, not pixels!)")
print("Images will be loaded on-the-fly in batches during training via tf.data")


## 2. Visualizing Sample Images

Let me look at what concrete crack images and non-crack images look like.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

# show 5 crack images
crack_paths_sample = [p for p, l in zip(all_paths, all_labels) if l == 1][:5]
for i, path in enumerate(crack_paths_sample):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    axes[0, i].imshow(img)
    axes[0, i].set_title('Crack (Positive)')
    axes[0, i].axis('off')

# show 5 no-crack images
no_crack_paths_sample = [p for p, l in zip(all_paths, all_labels) if l == 0][:5]
for i, path in enumerate(no_crack_paths_sample):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    axes[1, i].imshow(img)
    axes[1, i].set_title('No Crack (Negative)')
    axes[1, i].axis('off')

plt.suptitle('Sample Concrete Surface Images', fontsize=14)
plt.tight_layout()
save_fig('sample_images')
plt.show()


In [ ]:
labels_names = ['No Crack', 'Crack']
counts = [len(all_labels) - sum(all_labels), sum(all_labels)]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(labels_names, counts, color=['#4CAF50', '#F44336'])
ax.set_title('Dataset Class Distribution')
ax.set_ylabel('Number of Images')
for i, v in enumerate(counts):
    ax.text(i, v + 200, str(v), ha='center', fontsize=12)

plt.tight_layout()
save_fig('class_distribution')
plt.show()

print(f"The dataset is perfectly balanced: {counts[0]} vs {counts[1]}")


## 3. Preprocessing and Data Splitting

Since the dataset doesn't have a predefined split, I'll create my own: 70% training, 15% validation, 15% test. The dataset is already balanced so no special handling needed.

In [ ]:
all_labels_arr = np.array(all_labels)

# first split: 85% train+val, 15% test  (splitting paths, not pixel arrays!)
train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
    all_paths, all_labels_arr, test_size=0.15, random_state=42, stratify=all_labels_arr
)

# second split: from trainval, take ~17.6% as val (= 15% of total)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_val_paths, train_val_labels, test_size=0.176, random_state=42, stratify=train_val_labels
)

print(f"Training set:   {len(train_paths)} images")
print(f"Validation set: {len(val_paths)} images")
print(f"Test set:       {len(test_paths)} images")
print(f"\nTraining   - No Crack: {sum(train_labels==0)}, Crack: {sum(train_labels==1)}")
print(f"Validation - No Crack: {sum(val_labels==0)}, Crack: {sum(val_labels==1)}")
print(f"Test       - No Crack: {sum(test_labels==0)}, Crack: {sum(test_labels==1)}")


## 4. Data Augmentation

Even though the dataset is balanced and large, augmentation will help the model generalize better. For crack detection, both horizontal AND vertical flips make sense because cracks can appear in any orientation — unlike medical images where flipping upside down would be wrong.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess(path, label):
    """Read image file, decode, resize, normalize — runs on GPU/CPU on the fly"""
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

def augment_image(img, label):
    """Augmentation applied only to training data — cracks can appear at any angle"""
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)          # valid for cracks
    img = tf.image.random_brightness(img, max_delta=0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label

def make_dataset(paths, labels, augment_data=False):
    ds = tf.data.Dataset.from_tensor_slices((list(paths), labels))
    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    if augment_data:
        ds = ds.map(augment_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_paths, train_labels, augment_data=True)
val_ds   = make_dataset(val_paths,   val_labels,   augment_data=False)
test_ds  = make_dataset(test_paths,  test_labels,  augment_data=False)

print("tf.data pipelines created — images load on-the-fly, no RAM spike!")
print(f"  train_ds: {len(train_paths)} images")
print(f"  val_ds:   {len(val_paths)} images")
print(f"  test_ds:  {len(test_paths)} images")

# visualize augmentation on one sample crack image
sample_path = [p for p, l in zip(train_paths, train_labels.tolist()) if l == 1][0]
sample_img_raw, _ = load_and_preprocess(sample_path, 1)

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
axes[0].imshow(sample_img_raw.numpy())
axes[0].set_title('Original')
axes[0].axis('off')

for i in range(1, 6):
    aug_img, _ = augment_image(sample_img_raw, 1)
    axes[i].imshow(aug_img.numpy())
    axes[i].set_title(f'Augmented {i}')
    axes[i].axis('off')

plt.suptitle('Data Augmentation Examples (Crack Image)', fontsize=13)
plt.tight_layout()
save_fig('augmentation_examples')
plt.show()


## 5. Building the Custom CNN

Starting with a CNN built from scratch. 4 convolutional blocks with batch normalization. This will serve as the baseline to compare against the pretrained models.

In [ ]:
def build_custom_cnn():
    model = models.Sequential([
        # block 1
        layers.Conv2D(32, (3, 3), padding='same', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),

        # block 2
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),

        # block 3
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),

        # block 4
        layers.Conv2D(256, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),

        # classifier head
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

custom_model = build_custom_cnn()
custom_model.summary()

In [ ]:
custom_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
    ModelCheckpoint('best_custom_cnn.keras', monitor='val_accuracy', save_best_only=True)
]

EPOCHS = 30

print("Training Custom CNN...")
cnn_start = time.time()

custom_history = custom_model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks
)

custom_train_time = time.time() - cnn_start
print(f"\nTraining time: {custom_train_time/60:.1f} minutes")


## 6. Evaluating the Custom CNN

Let me check how the custom model performs on the test set.

In [ ]:
def plot_training_curves(history, model_name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(history.history['accuracy'], label='Train')
    ax1.plot(history.history['val_accuracy'], label='Validation')
    ax1.set_title(f'{model_name} - Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(history.history['loss'], label='Train')
    ax2.plot(history.history['val_loss'], label='Validation')
    ax2.set_title(f'{model_name} - Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    save_fig(f'{model_name.lower().replace(" ", "_")}_curves')
    plt.show()


def evaluate_model(model, test_ds, test_labels, model_name):
    y_test = np.array(test_labels)
    y_pred_prob = model.predict(test_ds, verbose=1).flatten()
    y_pred = (y_pred_prob > 0.5).astype(int)

    print(f"\n{'='*50}")
    print(f"Results for {model_name}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=['No Crack', 'Crack']))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Crack', 'Crack'],
                yticklabels=['No Crack', 'Crack'])
    plt.title(f'{model_name} - Confusion Matrix')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    save_fig(f'{model_name.lower().replace(" ", "_")}_cm')
    plt.show()

    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    print(f"Test Accuracy: {test_acc:.4f}")

    fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
    roc_auc = auc(fpr, tpr)
    print(f"AUC-ROC: {roc_auc:.4f}")

    # inference time — measure on one batch
    one_batch = next(iter(test_ds.take(1)))[0]
    start = time.time()
    _ = model.predict(one_batch, verbose=0)
    inf_time = (time.time() - start) / len(one_batch) * 1000
    print(f"Inference time: {inf_time:.2f} ms/image")

    return y_pred, y_pred_prob, fpr, tpr, roc_auc, inf_time


plot_training_curves(custom_history, 'Custom CNN')
custom_pred, custom_prob, custom_fpr, custom_tpr, custom_auc, custom_inf = evaluate_model(
    custom_model, test_ds, test_labels, 'Custom CNN'
)


## 7. Transfer Learning — MobileNetV2

Now using MobileNetV2 pretrained on ImageNet. I'll freeze the base first and train only the top layers, then fine-tune some of the later layers with a much lower learning rate.

In [ ]:
def build_mobilenet():
    base_model = MobileNetV2(weights='imagenet', include_top=False,
                              input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base_model.trainable = False

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    return model, base_model

mobilenet_model, mobilenet_base = build_mobilenet()
mobilenet_model.summary()
print(f"\nTrainable parameters: {sum([tf.keras.backend.count_params(w) for w in mobilenet_model.trainable_weights]):,}")

In [ ]:
mobilenet_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks_mobile = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
]

print("Phase 1: Training with frozen base layers...")
mobile_start = time.time()

mobile_history_1 = mobilenet_model.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds,
    callbacks=callbacks_mobile
)


In [ ]:
# unfreeze last 20 layers for fine-tuning
mobilenet_base.trainable = True
for layer in mobilenet_base.layers[:-20]:
    layer.trainable = False

# recompile with much lower learning rate
mobilenet_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks_mobile_ft = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
    ModelCheckpoint('best_mobilenet.keras', monitor='val_accuracy', save_best_only=True)
]

print("Phase 2: Fine-tuning last 20 layers...")
mobile_history_2 = mobilenet_model.fit(
    train_ds,
    epochs=15,
    validation_data=val_ds,
    callbacks=callbacks_mobile_ft
)

mobile_train_time = time.time() - mobile_start
print(f"\nTotal MobileNetV2 training time: {mobile_train_time/60:.1f} minutes")


In [ ]:
# combine both training phases for plotting
mobile_history_combined = {}
for key in mobile_history_1.history:
    mobile_history_combined[key] = mobile_history_1.history[key] + mobile_history_2.history[key]

class CombinedHistory:
    def __init__(self, history_dict):
        self.history = history_dict

plot_training_curves(CombinedHistory(mobile_history_combined), 'MobileNetV2')

mobile_pred, mobile_prob, mobile_fpr, mobile_tpr, mobile_auc, mobile_inf = evaluate_model(
    mobilenet_model, test_ds, test_labels, 'MobileNetV2'
)


## 8. Transfer Learning — ResNet50

Trying ResNet50 as the second transfer learning model. Same two-phase approach — frozen base first, then fine-tune.

In [ ]:
def build_resnet():
    base_model = ResNet50(weights='imagenet', include_top=False,
                          input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base_model.trainable = False

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    return model, base_model

resnet_model, resnet_base = build_resnet()

# phase 1: frozen base
resnet_model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

resnet_callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
]

print("Phase 1: Training ResNet50 with frozen base...")
resnet_start = time.time()

resnet_history_1 = resnet_model.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds,
    callbacks=resnet_callbacks
)

# phase 2: fine-tune last 20 layers
resnet_base.trainable = True
for layer in resnet_base.layers[:-20]:
    layer.trainable = False

resnet_model.compile(optimizer=Adam(learning_rate=1e-5), loss='binary_crossentropy', metrics=['accuracy'])

resnet_callbacks_ft = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
    ModelCheckpoint('best_resnet50.keras', monitor='val_accuracy', save_best_only=True)
]

print("\nPhase 2: Fine-tuning ResNet50...")
resnet_history_2 = resnet_model.fit(
    train_ds,
    epochs=15,
    validation_data=val_ds,
    callbacks=resnet_callbacks_ft
)

resnet_train_time = time.time() - resnet_start
print(f"\nTotal ResNet50 training time: {resnet_train_time/60:.1f} minutes")


In [ ]:
resnet_history_combined = {}
for key in resnet_history_1.history:
    resnet_history_combined[key] = resnet_history_1.history[key] + resnet_history_2.history[key]

plot_training_curves(CombinedHistory(resnet_history_combined), 'ResNet50')

resnet_pred, resnet_prob, resnet_fpr, resnet_tpr, resnet_auc, resnet_inf = evaluate_model(
    resnet_model, test_ds, test_labels, 'ResNet50'
)


## 9. Model Comparison

Putting all results side by side to see which model works best for crack detection.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_test = np.array(test_labels)

results = {}
for name, y_pred, y_prob, model, inf_time in [
    ('Custom CNN', custom_pred, custom_prob, custom_model, custom_inf),
    ('MobileNetV2', mobile_pred, mobile_prob, mobilenet_model, mobile_inf),
    ('ResNet50', resnet_pred, resnet_prob, resnet_model, resnet_inf)
]:
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'AUC-ROC': auc(roc_curve(y_test, y_prob)[0], roc_curve(y_test, y_prob)[1]),
        'Parameters': model.count_params(),
        'Inference (ms)': round(inf_time, 2)
    }

results_df = pd.DataFrame(results).T
print(results_df.to_string())

results_df.to_csv('model_comparison.csv')
print("\nResults saved to model_comparison.csv")


In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
models_list = list(results.keys())

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics))
width = 0.25

for i, model_name in enumerate(models_list):
    values = [results[model_name][m] for m in metrics]
    ax.bar(x + i*width, values, width, label=model_name)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Concrete Crack Detection')
ax.legend()
ax.set_ylim(0.8, 1.0)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
save_fig('model_comparison')
plt.show()


## 10. ROC Curve Comparison

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(custom_fpr, custom_tpr, label=f'Custom CNN (AUC = {custom_auc:.4f})')
plt.plot(mobile_fpr, mobile_tpr, label=f'MobileNetV2 (AUC = {mobile_auc:.4f})')
plt.plot(resnet_fpr, resnet_tpr, label=f'ResNet50 (AUC = {resnet_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison — Crack Detection')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
save_fig('roc_comparison')
plt.show()


## 11. Grad-CAM Visualization

Grad-CAM shows what parts of the image the model is looking at when making a prediction. For crack detection, we want to see the model focusing on the actual crack lines, not random texture or background patterns.

In [ ]:
def load_single_image(path):
    """Load one image from path for visualization/Grad-CAM (not RAM-intensive)"""
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    return img / 255.0


def simple_gradcam(model, image):
    """Grad-CAM for sequential model with transfer learning base"""
    img_tensor = tf.cast(image[np.newaxis], tf.float32)

    base = model.layers[0]  # the pretrained base

    # find last conv layer in base
    last_conv_name = None
    for layer in reversed(base.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            last_conv_name = layer.name
            break

    mini_model = tf.keras.Model(
        inputs=base.input,
        outputs=[base.get_layer(last_conv_name).output, base.output]
    )

    with tf.GradientTape() as tape:
        conv_out, base_out = mini_model(img_tensor)
        x = base_out
        for layer in model.layers[1:]:
            x = layer(x)
        loss = x[:, 0]

    grads = tape.gradient(loss, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))

    heatmap = conv_out[0] @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_gradcam(img, heatmap, alpha=0.4):
    """Overlay heatmap on original image"""
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    superimposed = np.uint8(heatmap * alpha + img * 255 * (1 - alpha))
    return superimposed


In [ ]:
# use the best transfer learning model for grad-cam
print(f"Best model: {results_df['Accuracy'].astype(float).idxmax()}")
gradcam_model = mobilenet_model  # using MobileNetV2

correct_crack    = np.where((test_labels == 1) & (mobile_pred == 1))[0][:3]
correct_no_crack = np.where((test_labels == 0) & (mobile_pred == 0))[0][:3]
misclassified    = np.where(test_labels != mobile_pred)[0][:3]

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
row_labels = ['Correct: Crack', 'Correct: No Crack', 'Misclassified']
indices_list = [correct_crack, correct_no_crack, misclassified]

for row, (label, indices) in enumerate(zip(row_labels, indices_list)):
    for col, idx in enumerate(indices[:3]):
        img = load_single_image(test_paths[idx])   # load just this one image
        heatmap = simple_gradcam(gradcam_model, img)
        overlay = overlay_gradcam(img, heatmap)

        axes[row, col*2].imshow(img)
        true_label = 'Crack' if test_labels[idx] else 'No Crack'
        axes[row, col*2].set_title(f'{label}\nTrue: {true_label}')
        axes[row, col*2].axis('off')

        axes[row, col*2+1].imshow(overlay)
        pred_label = 'Crack' if mobile_pred[idx] else 'No Crack'
        axes[row, col*2+1].set_title(f'Grad-CAM\nPred: {pred_label}')
        axes[row, col*2+1].axis('off')

plt.suptitle('Grad-CAM Visualizations — Where the Model Looks', fontsize=14)
plt.tight_layout()
save_fig('gradcam_results')
plt.show()


## 12. SHAP Analysis — Pixel-Level Feature Importance

Grad-CAM shows *where* the model attends; SHAP explains *why* a specific prediction was made in terms of individual pixel contributions.  
We use `shap.GradientExplainer` on **MobileNetV2** and visualize:
- **Row 1**: Original image with true/predicted labels
- **Row 2**: Positive SHAP overlay — pixels that *push the prediction toward "Crack"*
- **Row 3**: Total SHAP magnitude — pixels with highest absolute influence (regardless of direction)


In [ ]:
!pip install shap -q
import shap

# ── Background set: 50 images sampled from test_ds (numpy, needed by GradientExplainer) ──
bg_imgs = []
for imgs, _ in test_ds.take(2):          # 2 batches × 32 = 64 images
    bg_imgs.append(imgs.numpy())
background = np.concatenate(bg_imgs, axis=0)[:50]   # shape: (50, 224, 224, 3)

# ── 4 sample images: 2 crack + 2 no-crack ──
crack_idxs    = np.where(test_labels == 1)[0][:2]
no_crack_idxs = np.where(test_labels == 0)[0][:2]
sample_idxs   = list(crack_idxs) + list(no_crack_idxs)

sample_imgs = np.array([load_single_image(test_paths[i]) for i in sample_idxs])  # (4, 224, 224, 3)
sample_lbls = [int(test_labels[i]) for i in sample_idxs]

# ── GradientExplainer ──
print("Computing SHAP values (GradientExplainer on MobileNetV2) — may take ~1–2 min on CPU…")
explainer   = shap.GradientExplainer(mobilenet_model, background)
shap_values = explainer.shap_values(sample_imgs)   # list with one array for binary output

# shap_values[0] → shape (4, 224, 224, 3)  — contribution of each pixel-channel to Crack class
sv = np.array(shap_values[0])

# ── Visualize: 3 rows × 4 columns ──
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
class_names = ['No Crack', 'Crack']

for i in range(4):
    img      = sample_imgs[i]
    sv_img   = sv[i]                                               # (224, 224, 3)
    pred_prob = float(mobilenet_model.predict(img[np.newaxis], verbose=0)[0][0])
    pred_lbl  = class_names[int(pred_prob > 0.5)]
    true_lbl  = class_names[sample_lbls[i]]

    # Row 0 — original image
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'True: {true_lbl}\nPred: {pred_lbl} ({pred_prob:.2f})', fontsize=10)
    axes[0, i].axis('off')

    # Row 1 — positive SHAP (pixels pushing toward Crack)
    shap_pos = np.maximum(sv_img, 0).sum(axis=-1)                 # (224, 224)
    shap_pos = shap_pos / (shap_pos.max() + 1e-8)
    axes[1, i].imshow(img)
    axes[1, i].imshow(shap_pos, cmap='Reds', alpha=0.6)
    axes[1, i].set_title('Positive SHAP\n(pushes → Crack)', fontsize=9)
    axes[1, i].axis('off')

    # Row 2 — total SHAP magnitude
    shap_mag = np.abs(sv_img).sum(axis=-1)
    shap_mag = shap_mag / (shap_mag.max() + 1e-8)
    axes[2, i].imshow(img)
    axes[2, i].imshow(shap_mag, cmap='hot', alpha=0.6)
    axes[2, i].set_title('Total SHAP Importance', fontsize=9)
    axes[2, i].axis('off')

plt.suptitle(
    'SHAP Analysis — Pixel Importance for MobileNetV2\n'
    '(Red/Hot = strong influence on crack prediction)',
    fontsize=13
)
plt.tight_layout()
save_fig('shap_analysis')
plt.show()

print("\nInterpretation:")
print("- Positive SHAP (Reds): pixel regions that strongly signal CRACK presence")
print("- Total magnitude (Hot): pixels the model relies on most regardless of direction")
print("- If overlays appear diffuse, the model uses distributed texture — consistent with surface crack patterns")


## 13. Error Analysis

Looking at what the model gets wrong. Since the dataset is balanced, errors should be roughly split between false positives and false negatives.


In [ ]:
misclassified_idx = np.where(test_labels != mobile_pred)[0]
print(f"Total misclassified: {len(misclassified_idx)} out of {len(test_labels)} ({len(misclassified_idx)/len(test_labels)*100:.1f}%)")

fp_idx = np.where((test_labels == 0) & (mobile_pred == 1))[0]  # no crack → predicted crack
fn_idx = np.where((test_labels == 1) & (mobile_pred == 0))[0]  # crack → predicted no crack

print(f"False Positives (No Crack predicted as Crack): {len(fp_idx)}")
print(f"False Negatives (Crack predicted as No Crack): {len(fn_idx)}")

fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i in range(min(5, len(fp_idx))):
    img = load_single_image(test_paths[fp_idx[i]])   # load just this image
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'FP (conf: {mobile_prob[fp_idx[i]]:.2f})')
    axes[0, i].axis('off')
for i in range(min(5, len(fp_idx)), 5):
    axes[0, i].axis('off')
axes[0, 0].set_ylabel('False Positives\n(No Crack → Crack)', fontsize=9)

for i in range(min(5, len(fn_idx))):
    img = load_single_image(test_paths[fn_idx[i]])
    axes[1, i].imshow(img)
    axes[1, i].set_title(f'FN (conf: {mobile_prob[fn_idx[i]]:.2f})')
    axes[1, i].axis('off')
for i in range(min(5, len(fn_idx)), 5):
    axes[1, i].axis('off')
axes[1, 0].set_ylabel('False Negatives\n(Crack → No Crack)', fontsize=9)

plt.suptitle('Error Analysis — Misclassified Images (MobileNetV2)', fontsize=13)
plt.tight_layout()
save_fig('error_analysis')
plt.show()

print("\nObservations:")
print("- False positives may be surfaces with textures or patterns that look like cracks")
print("- False negatives are likely images with very thin or hairline cracks")


## 13. Final Summary

In [ ]:
print("\n" + "="*60)
print("FINAL MODEL COMPARISON — CONCRETE CRACK DETECTION")
print("="*60)
print(results_df[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']].to_string())
print("="*60)
print(f"\nBest model: {results_df['Accuracy'].astype(float).idxmax()}")
print(f"Best accuracy: {results_df['Accuracy'].astype(float).max():.4f}")
print(f"Best AUC: {results_df['AUC-ROC'].astype(float).max():.4f}")

print("\nTraining times:")
print(f"  Custom CNN:  {custom_train_time/60:.1f} min")
print(f"  MobileNetV2: {mobile_train_time/60:.1f} min")
print(f"  ResNet50:    {resnet_train_time/60:.1f} min")

print("\nSaved figures:")
import glob
for f in sorted(glob.glob('*.png')):
    print(f"  - {f}")

## Conclusion

All three models achieved outstanding results on the crack detection task, significantly exceeding initial expectations. The custom CNN reached 99.90% accuracy and AUC = 0.9999 — surprising given it was built from scratch with only 456K parameters. MobileNetV2 marginally edged it out at 99.93% accuracy with a perfect AUC of 1.0000, while ResNet50 trailed slightly at 99.02%, likely due to its larger capacity requiring more data/epochs to fully converge.

**Key observation from training curves:** The Custom CNN showed large spikes in validation loss during training (loss reaching ~3.5 at epoch 5), but `EarlyStopping` with `restore_best_weights=True` successfully recovered the best checkpoint, explaining the high final test accuracy despite unstable training. MobileNetV2 showed clean progressive improvement during fine-tuning. ResNet50 showed the characteristic loss spike at epoch 10 when the base layers were unfrozen — expected behavior when switching from frozen to fine-tuning phase.

**Grad-CAM observation:** For correctly classified crack images, the heatmaps appear mostly uniform (blue), suggesting MobileNetV2 uses distributed global texture features rather than highlighting specific crack lines. The misclassified cases show concentrated red activations on surface patterns that visually resemble cracks (construction joints, water stains) or on areas adjacent to the actual crack — confirming the model's decision boundary relies on surface texture statistics rather than explicit crack geometry.

For real-world use in bridge or building inspection, this system would work well as an automated screening tool. The Custom CNN's efficiency (19.8 ms/image vs 129 ms for MobileNetV2) makes it preferable for real-time drone imagery, while MobileNetV2 is better for batch processing where accuracy is critical.